In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
from ultralytics import YOLO
from src.config import DATASET_ROOT, POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_pose_tracking_video, extract_pose_tracking_data, save_multiple_tracked_video, pose_data_to_stgcn_tensor
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
from scripts.common.get_device import get_available_device
from tqdm import tqdm
from pathlib import Path
import gc
import torch
from torch.utils.data import Dataset

gc.collect()
torch.cuda.empty_cache()


In [37]:
pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split='train')
video_dataset = RWF2000Dataset(DATASET_ROOT, split="train")

In [38]:
video_dataset.samples[0]

(PosixPath('/homes/mp2940/demo/datasets/rwf-2000/RWF-2000/train/NonFight/-1l5631l3fg_0.avi'),
 0)

In [35]:
validate_pose_dataset(pose_dataset)

100%|██████████| 1600/1600 [00:38<00:00, 41.34it/s]

Dataset size: 1600
NonFight samples: 800
Fight samples: 800

Empty samples
NonFight: 200
Fight: 36
Total: 236


[PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/0AvTZRYx_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/11vr9Jho_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/16O0PIQP_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/1ahhhDBQHxg_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/1dsLuL5Lvbc_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/1r91ACCe_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/2KvkZ4yo_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/3HCpeji1_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/3TQqO4N8_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/3Vw7MoNBgx4_0.pt'),
 PosixPath('/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/NonFight/3Vw7MoN

In [26]:
pose_dataset[0][1]

tensor(0)

In [3]:
# pose data testing
pose_file = Path("/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/Fight/0_DzLlklZa0_3.pt")
pose_data = torch.load(pose_file, weights_only=False)

In [4]:
tensor = pose_data_to_stgcn_tensor(pose_data)


In [6]:
tensor.shape

torch.Size([3, 150, 17, 2])

In [ ]:
train_dataset = RWF2000Dataset(
    DATASET_ROOT, 
    split="train", 
    return_rgb_frames=False
)

val_dataset = RWF2000Dataset(
    DATASET_ROOT, 
    split="val", 
    return_rgb_frames=False
)

device = str(get_available_device())
model_path = "yolo26l-pose.pt"
model = YOLO(model_path)

In [ ]:
# save a slice of tracked video .mp4 files
#save_multiple_tracked_video(0, 10, model_path, train_dataset)

In [ ]:
# view a slice of pose data from dataset

start, stop = 0, 10

test_videos = train_dataset.samples[start:stop]
device = str(get_available_device())
results = []
for path, label in test_videos:
    video_pose_data = extract_pose_tracking_data(video_path=path,
                                                 model=model,
                                                 device=device)
    results.append(video_pose_data)

In [ ]:
first_result = results[0]

In [3]:
# pose data testing
pose_file = Path("/homes/mp2940/demo/datasets/rwf-2000/pose_data/train/Fight/0_DzLlklZa0_3.pt")
pose_data = torch.load(pose_file, weights_only=False)

In [5]:
pose_data

{'video_path': '/homes/mp2940/demo/datasets/rwf-2000/RWF-2000/train/Fight/0_DzLlklZa0_3.avi',
 'original_video_shape': (720, 1280),
 'frames': [{'frame_index': 0, 'people': []},
  {'frame_index': 1,
   'people': [{'track_id': 7051,
     'bbox': tensor([873.4679, 259.1725, 993.1938, 497.4795]),
     'bbox_confidence': 0.8555864691734314,
     'keypoints': tensor([[977.2860, 283.5587],
             [974.7819, 277.8964],
             [980.1934, 280.5746],
             [960.2352, 277.0934],
             [982.6948, 284.8987],
             [943.1204, 299.0491],
             [981.8561, 317.7302],
             [928.6514, 325.1356],
             [978.7418, 357.5014],
             [940.1378, 338.0394],
             [970.2584, 369.0046],
             [930.7369, 371.4445],
             [957.3016, 379.8514],
             [914.0918, 411.6142],
             [955.0885, 428.6370],
             [893.3926, 451.3222],
             [951.0482, 474.8986]]),
     'keypoint_confidence': tensor([0.0138, 0.0041,

In [18]:
for person in pose_data["frames"][1]['people']:
    print(type(person['keypoint_confidence'].float()[0]))

<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [4]:
pose_data_to_stgcn_tensor(pose_data)

TypeError: unsupported operand type(s) for +=: 'collections.defaultdict' and 'int'